In [ ]:
!pip install dandi

# Download the entire dataset into the Colab environment
!dandi download DANDI:000008/0.211014.0809 --output-dir /content/dandi_000008

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.9/370.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.6/118.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.9/341.9 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 12.9 MB/s eta 0:00:00
   ━━━━

In [ ]:
!mkdir -p /content/dandi_000008
!dandi download DANDI:000008/0.211014.0809 --output-dir /content/dandi_000008

PATH                                                                                                                      SIZE       DONE            DONE% CHECKSUM STATUS          MESSAGE   
000008/dandiset.yaml                                                                                                                                                done            updated   
000008/sub-mouse-AAYYT/sub-mouse-AAYYT_ses-20180420-sample-2_slice-20180420-slice-2_cell-20180420-sample-2_icephys.nwb    9.3 MB     9.3 MB           100%    ok    done                      
000008/sub-mouse-AAYYT/sub-mouse-AAYYT_ses-20180420-sample-3_slice-20180420-slice-3_cell-20180420-sample-3_icephys.nwb    9.3 MB     9.3 MB           100%    ok    done                      
000008/sub-mouse-AAYYT/sub-mouse-AAYYT_ses-20180420-sample-4_slice-20180420-slice-4_cell-20180420-sample-4_icephys.nwb    9.5 MB     9.5 MB           100%    ok    done                      
000008/sub-mouse-AEJGZ/sub-mouse-AEJGZ_ses-20

In [ ]:
# ============================================================
# PREPROCESSING FILTER:
# Keep only cells that have *ALL* sweeps 10–70 inclusive,
# AND each sweep is long enough for CUT_END (so cropping works).
# Save the kept file paths as a .npy array (no CSV).
# ============================================================

import os, re, h5py
import numpy as np

ROOT_DIR    = "/content/dandi_000008"
SWEEP_START = 10
SWEEP_END   = 70
REQ_SWEEPS  = list(range(SWEEP_START, SWEEP_END + 1))

CUT_START   = 2500
CUT_END     = 17500  # must have len >= CUT_END

OUT_NPY     = f"/content/kept_cells_sweeps{SWEEP_START:03d}_{SWEEP_END:03d}_cut{CUT_START}_{CUT_END}.npy"

# ----------------------------
# Helpers
# ----------------------------
def find_nwb_files(root):
    out = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.endswith(".nwb"):
                out.append(os.path.join(dirpath, fn))
    return sorted(out)

def has_sweep_and_long_enough(f, sidx, cut_end):
    acq = f.get("acquisition", None)
    if acq is None:
        return False, "no_acquisition"
    key = f"CurrentClampSeries{sidx:03d}"
    if key not in acq:
        return False, "missing_sweep"
    vlen = int(acq[key]["data"].shape[0])
    if vlen < cut_end:
        return False, "too_short"
    return True, None

# ----------------------------
# Scan all NWB files
# ----------------------------
nwb_files = find_nwb_files(ROOT_DIR)
print("Found NWB files:", len(nwb_files))
print(f"Required sweeps: {SWEEP_START}–{SWEEP_END} (n={len(REQ_SWEEPS)})")
print(f"Crop window: [{CUT_START}:{CUT_END}] → need each sweep length ≥ {CUT_END}")

kept_paths = []
skip = {"no_acquisition": 0, "missing_sweep": 0, "too_short": 0, "h5_error": 0, "other": 0}

for fp in nwb_files:
    try:
        with h5py.File(fp, "r") as f:
            ok = True
            first_fail_reason = None
            for sidx in REQ_SWEEPS:
                good, reason = has_sweep_and_long_enough(f, sidx, CUT_END)
                if not good:
                    ok = False
                    first_fail_reason = reason
                    break

            if ok:
                kept_paths.append(fp)
            else:
                if first_fail_reason in skip:
                    skip[first_fail_reason] += 1
                else:
                    skip["other"] += 1
    except Exception:
        skip["h5_error"] += 1

# ----------------------------
# Save kept list as .npy
# ----------------------------
kept_paths = np.array(kept_paths)
np.save(OUT_NPY, kept_paths)

print("\n================ SUMMARY ================")
print("Total NWB files scanned:", len(nwb_files))
print("Cells kept (have ALL sweeps 10–70, long enough):", len(kept_paths))
print("\nSkip counts (first failure per file):")
for k, v in skip.items():
    print(f"  {k}: {v}")

print("\n✅ Saved kept file paths to:", OUT_NPY)


Found NWB files: 1328
Required sweeps: 10–70 (n=61)
Crop window: [2500:17500] → need each sweep length ≥ 17500

================ SUMMARY ================
Total NWB files scanned: 1328
Cells kept (have ALL sweeps 10–70, long enough): 1264

Skip counts (first failure per file):
  no_acquisition: 0
  missing_sweep: 64
  too_short: 0
  h5_error: 0
  other: 0

✅ Saved kept file paths to: /content/kept_cells_sweeps010_070_cut2500_17500.npy


In [ ]:
!pip install kymatio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 4.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 2 (AFTER YOUR FILTER):
# Load kept_paths (.npy)  -> for each kept cell:
#   - crop sweeps 10..70 at [CUT_START:CUT_END]
#   - stack in sweep order => fixed length trace (61 * SEG_LEN)
#   - z-score per cell
#   - compute WST with J=12, Q=20, max_order=2
#   - save ONE NPZ PER CELL with proper identifiers for later joins
# Also:
#   - make WST+trace figures for 9 random cells
# ============================================================

import os, re, random
import numpy as np
import h5py
import matplotlib.pyplot as plt

import torch
from kymatio.torch import Scattering1D

# ----------------------------
# INPUT: the file you already created
# ----------------------------
KEPT_NPY = "/content/kept_cells_sweeps010_070_cut2500_17500.npy"

# ----------------------------
# CONFIG (must match filter)
# ----------------------------
SWEEP_START = 10
SWEEP_END   = 70
REQ_SWEEPS  = list(range(SWEEP_START, SWEEP_END + 1))

CUT_START = 2500
CUT_END   = 17500
SEG_LEN   = CUT_END - CUT_START

# WST params (as requested)
J = 12
Q = 20
OVERSAMPLING = 0
MAX_ORDER = 2

# Outputs
OUTDIR = f"/content/WST_kept_cells_sweeps{SWEEP_START:03d}_{SWEEP_END:03d}_cut{CUT_START}_{CUT_END}_J{J}_Q{Q}"
PER_CELL_DIR = os.path.join(OUTDIR, "per_cell_npz")
FIG_DIR      = os.path.join(OUTDIR, "figures_random9")

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(PER_CELL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Random plots
N_RANDOM_PLOTS = 9
RANDOM_SEED = 42

# ----------------------------
# Helpers
# ----------------------------
def file_to_cell_id(fp: str) -> str:
    return os.path.splitext(os.path.basename(fp))[0]

def cell_id_to_short_id(cell_id: str):
    m = re.search(r"ses-(\d+)-sample-(\d+)", str(cell_id))
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

def safe_name(x: str) -> str:
    return re.sub(r"[^\w\-_\.]+", "_", str(x))

def zscore_1d(x, eps=1e-8):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + eps)

def save_fig(path, dpi=220):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

def plot_one_cell_bundle(out_subdir, name, stacked, seg_len,
                         S0, S1, S2, xi1_sorted, xi_pairs_sorted, parent_change_rows):
    os.makedirs(out_subdir, exist_ok=True)

    # stacked trace + boundaries
    plt.figure(figsize=(18, 4))
    plt.plot(stacked, lw=0.8)
    for k in range(1, (SWEEP_END - SWEEP_START + 1)):
        plt.axvline(k * seg_len, color="k", alpha=0.15, lw=1)
    plt.title(f"Stacked voltage | {name} | sweeps {SWEEP_START}-{SWEEP_END} | cut[{CUT_START}:{CUT_END}]")
    plt.xlabel("stacked index")
    plt.ylabel("voltage (raw units)")
    save_fig(os.path.join(out_subdir, "01_stacked_trace.png"))

    # S0
    plt.figure(figsize=(10, 2.6))
    plt.plot(S0, lw=1.2)
    plt.title("WST S0")
    plt.xlabel("WST time frames")
    plt.ylabel("S0")
    save_fig(os.path.join(out_subdir, "02_S0.png"))

    # S1 heatmap (y-axis = xi1)
    plt.figure(figsize=(11, 4.5))
    plt.imshow(S1, aspect="auto", origin="upper", cmap="jet")
    ax = plt.gca()
    K = len(xi1_sorted)
    step = max(1, K // 10)
    ticks = np.arange(0, K, step)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{xi1_sorted[t]:.3f}" for t in ticks])
    plt.title("WST S1 (sorted by xi1 desc)")
    plt.xlabel("WST time frames")
    plt.ylabel("xi1")
    plt.colorbar()
    save_fig(os.path.join(out_subdir, "03_S1.png"))

    # S2 heatmap (normalize per-parent for visualization)
    S2n = S2.copy()
    start = 0
    for brk in list(parent_change_rows) + [S2n.shape[0]]:
        block = S2n[start:brk]
        d = np.max(block)
        if d > 0:
            S2n[start:brk] = block / d
        start = brk

    plt.figure(figsize=(12, 5.5))
    plt.imshow(S2n, aspect="auto", origin="upper", cmap="jet")
    ax = plt.gca()
    for r in parent_change_rows:
        ax.axhline(y=r - 0.5, color="white", linewidth=0.6, alpha=0.9)

    npaths = S2n.shape[0]
    step2 = max(1, npaths // 12)
    ticks2 = np.arange(0, npaths, step2)
    ax.set_yticks(ticks2)
    ax.set_yticklabels(
        [f"({xi_pairs_sorted[t,0]:.3f},{xi_pairs_sorted[t,1]:.3f})" for t in ticks2],
        fontsize=8
    )
    plt.title("WST S2 (sorted; per-parent normalized for visualization)")
    plt.xlabel("WST time frames")
    plt.ylabel("(xi1, xi2)")
    plt.colorbar()
    save_fig(os.path.join(out_subdir, "04_S2.png"))

# ----------------------------
# Load kept paths
# ----------------------------
kept_paths = np.load(KEPT_NPY, allow_pickle=True)
kept_paths = [str(p) for p in kept_paths]
N = len(kept_paths)
print("✅ Loaded kept cells:", N)

if N == 0:
    raise RuntimeError("kept_paths is empty. Check your filter output file.")

# Fixed stacked length since all sweeps exist for these cells
N_SWEEPS  = len(REQ_SWEEPS)
L_total   = N_SWEEPS * SEG_LEN
print("Sweeps used:", SWEEP_START, "to", SWEEP_END, "| count =", N_SWEEPS)
print("SEG_LEN:", SEG_LEN, "=> stacked length L_total:", L_total)

# ----------------------------
# Build WST object (fixed length!)
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

scattering = Scattering1D(
    J=J, shape=L_total, Q=Q,
    oversampling=OVERSAMPLING,
    max_order=MAX_ORDER
).to(device)

meta = scattering.meta()
order = meta["order"]
xi_all = meta["xi"]

order0 = np.where(order == 0)[0]
order1 = np.where(order == 1)[0]
order2 = np.where(order == 2)[0]

# sort S1 by xi1 desc
xi1 = xi_all[order1, 0]
sort1 = np.argsort(xi1)[::-1]
xi1_sorted = xi1[sort1].astype(np.float32)

# sort S2 by (xi1 desc, xi2 desc)
xi_pairs = xi_all[order2]
perm_S2 = np.lexsort((-xi_pairs[:, 1], -xi_pairs[:, 0]))
xi_pairs_sorted = xi_pairs[perm_S2].astype(np.float32)

parent_change_rows = (np.where(np.diff(xi_pairs_sorted[:, 0]) != 0)[0] + 1).astype(int)

# ----------------------------
# Main loop: crop sweeps, stack, WST, save NPZ
# ----------------------------
saved_npz_paths = []
bad_cells = 0

for i, fp in enumerate(kept_paths):
    cell_id = file_to_cell_id(fp)
    short_id = cell_id_to_short_id(cell_id)
    name = f"{cell_id}__sweeps{SWEEP_START:03d}_{SWEEP_END:03d}__cut{CUT_START}_{CUT_END}"

    try:
        segs = []
        with h5py.File(fp, "r") as f:
            acq = f["acquisition"]
            for sidx in REQ_SWEEPS:
                k = f"CurrentClampSeries{sidx:03d}"
                seg = acq[k]["data"][CUT_START:CUT_END].astype(np.float32)
                # should always be SEG_LEN due to filtering, but keep safe:
                if seg.shape[0] != SEG_LEN:
                    raise RuntimeError(f"Sweep {sidx:03d} seg_len {seg.shape[0]} != {SEG_LEN}")
                segs.append(seg)

        segs = np.stack(segs, axis=0)          # (61, SEG_LEN)
        stacked = segs.reshape(-1)             # (L_total,)
        x = zscore_1d(stacked)                 # z-score per cell

        x_torch = torch.from_numpy(x[None, None, :]).to(device)
        with torch.no_grad():
            Sx = scattering(x_torch).detach().cpu().numpy()

        if Sx.ndim == 4:
            Sx = Sx[:, 0]  # (1,C,T')
        if Sx.shape[1] != len(order):
            Sx = np.transpose(Sx, (0, 2, 1))

        S0 = Sx[:, order0, :]                  # (1,1,T')
        S1_uns = Sx[:, order1, :]
        S2_uns = Sx[:, order2, :]

        S1 = S1_uns[:, sort1, :].astype(np.float32)
        S2 = S2_uns[:, perm_S2, :].astype(np.float32)
        S0 = S0.astype(np.float32)

        out_npz = os.path.join(PER_CELL_DIR, f"{safe_name(name)}__WST.npz")
        np.savez_compressed(
            out_npz,
            name=np.array([name]),
            cell_id=np.array([cell_id]),
            short_id=np.array([short_id if short_id is not None else ""]),
            file_path=np.array([fp]),

            sweeps_used=np.array(REQ_SWEEPS, dtype=int),
            cut=np.array([CUT_START, CUT_END], dtype=int),
            seg_len=np.array([SEG_LEN], dtype=int),
            stacked_len=np.array([L_total], dtype=int),

            stacked_trace=stacked.astype(np.float32),  # raw stacked trace

            S0=S0[0],   # (1,T')
            S1=S1[0],   # (n1,T')
            S2=S2[0],   # (n2,T')

            xi1=xi1_sorted,
            xi_pairs=xi_pairs_sorted,
            parent_change_rows=parent_change_rows,

            # params
            J=np.array([J], dtype=int),
            Q=np.array([Q], dtype=int),
            oversampling=np.array([OVERSAMPLING], dtype=int),
            max_order=np.array([MAX_ORDER], dtype=int),
        )

        saved_npz_paths.append(out_npz)

    except Exception as e:
        bad_cells += 1
        if bad_cells <= 10:
            print("⚠️ Failed:", fp, "|", repr(e))
        continue

    if (i + 1) % 50 == 0 or (i + 1) == N:
        print(f"Processed {i+1}/{N} (saved {len(saved_npz_paths)} NPZ, failed {bad_cells})")

print("\n✅ Done WST for kept cells.")
print("Saved NPZ:", len(saved_npz_paths))
print("Failed:", bad_cells)
print("OUTDIR:", OUTDIR)
print("NPZ folder:", PER_CELL_DIR)

# ----------------------------
# Plot 9 random cells
# ----------------------------
if len(saved_npz_paths) > 0:
    random.seed(RANDOM_SEED)
    pick = random.sample(saved_npz_paths, k=min(N_RANDOM_PLOTS, len(saved_npz_paths)))

    for j, npz_path in enumerate(pick, 1):
        z = np.load(npz_path, allow_pickle=True)

        name = str(z["name"][0])
        stacked = z["stacked_trace"]
        S0 = z["S0"][0] if z["S0"].ndim == 2 else z["S0"]
        S1 = z["S1"]
        S2 = z["S2"]
        xi1_s = z["xi1"]
        xi_pairs_s = z["xi_pairs"]
        pcr = z["parent_change_rows"]

        out_subdir = os.path.join(FIG_DIR, f"{j:02d}__{safe_name(name)}")
        plot_one_cell_bundle(out_subdir, name, stacked, SEG_LEN,
                             S0, S1, S2, xi1_s, xi_pairs_s, pcr)

    print("\n✅ Saved random-9 figures to:", FIG_DIR)
else:
    print("\n⚠️ No NPZ files saved; skipping random-9 plots.")


✅ Loaded kept cells: 1264
Sweeps used: 10 to 70 | count = 61
SEG_LEN: 15000 => stacked length L_total: 915000
Device: cuda
Processed 50/1264 (saved 50 NPZ, failed 0)
Processed 100/1264 (saved 100 NPZ, failed 0)
Processed 150/1264 (saved 150 NPZ, failed 0)
Processed 200/1264 (saved 200 NPZ, failed 0)
Processed 250/1264 (saved 250 NPZ, failed 0)
Processed 300/1264 (saved 300 NPZ, failed 0)
Processed 350/1264 (saved 350 NPZ, failed 0)
Processed 400/1264 (saved 400 NPZ, failed 0)
Processed 450/1264 (saved 450 NPZ, failed 0)
Processed 500/1264 (saved 500 NPZ, failed 0)
Processed 550/1264 (saved 550 NPZ, failed 0)
Processed 600/1264 (saved 600 NPZ, failed 0)
Processed 650/1264 (saved 650 NPZ, failed 0)
Processed 700/1264 (saved 700 NPZ, failed 0)
Processed 750/1264 (saved 750 NPZ, failed 0)
Processed 800/1264 (saved 800 NPZ, failed 0)
Processed 850/1264 (saved 850 NPZ, failed 0)
Processed 900/1264 (saved 900 NPZ, failed 0)
Processed 950/1264 (saved 950 NPZ, failed 0)
Processed 1000/1264 (sav

In [ ]:
import os, re
import numpy as np
import pandas as pd

# ----------------------------
# PATHS
# ----------------------------
META_PATH = "/content/m1_patchseq_meta_data (1).csv"  # TAB-delimited
NPZ_DIR   = "/content/WST_kept_cells_sweeps010_070_cut2500_17500_J12_Q20/per_cell_npz"
OUTDIR    = "/content/dataset_splits_balanced"
os.makedirs(OUTDIR, exist_ok=True)

RANDOM_SEED = 42
TRAIN_FRAC  = 0.60
VAL_FRAC    = 0.10
TEST_FRAC   = 0.30

# ----------------------------
# Helpers (yours)
# ----------------------------
def meta_to_short_id(x: str):
    if not isinstance(x, str):
        return None
    m = re.search(r"(\d+)_sample_(\d+)", x.strip())
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

def load_rna_family_map(meta_path: str):
    meta = pd.read_csv(meta_path, sep="\t")
    if "Cell" not in meta.columns:
        raise ValueError("Metadata must contain a 'Cell' column.")
    rna_col = None
    for c in meta.columns:
        if c.strip().lower().replace(" ", "") == "rnafamily":
            rna_col = c
            break
    if rna_col is None:
        raise ValueError("Could not find 'RNA family' column in metadata.")

    meta = meta[["Cell", rna_col]].copy()
    meta["short_id"] = meta["Cell"].astype(str).apply(meta_to_short_id)
    meta = meta.dropna(subset=["short_id"]).copy()
    meta["RNA_family"] = meta[rna_col].astype(str).str.strip()
    return dict(zip(meta["short_id"].values, meta["RNA_family"].values))

def list_npz(npz_dir):
    return sorted([
        os.path.join(npz_dir, f)
        for f in os.listdir(npz_dir)
        if f.endswith(".npz")
    ])

def npz_to_short_id(npz_path: str):
    """
    Prefer reading short_id stored inside the NPZ (your pipeline saves it).
    Fallback: try regex from filename if needed.
    """
    try:
        z = np.load(npz_path, allow_pickle=True)
        if "short_id" in z:
            sid = str(z["short_id"][0])
            sid = sid.strip()
            if sid != "":
                return sid
    except Exception:
        pass

    # fallback from filename: try to find ses-<d>-sample-<d>
    base = os.path.basename(npz_path)
    m = re.search(r"ses-(\d+)-sample-(\d+)", base)
    if m:
        return f"{m.group(1)}_sample_{m.group(2)}"
    return None

# ----------------------------
# 1) Build table: npz_path + short_id + RNA_family
# ----------------------------
rna_map = load_rna_family_map(META_PATH)

rows = []
for p in list_npz(NPZ_DIR):
    sid = npz_to_short_id(p)
    fam = rna_map.get(sid, None) if sid is not None else None
    rows.append((p, sid, fam))

df = pd.DataFrame(rows, columns=["npz_path", "short_id", "RNA_family"])

# keep only those with a valid family
df = df.dropna(subset=["short_id", "RNA_family"]).copy()

# ----------------------------
# 2) Remove "low quality"
# ----------------------------
# robust match (case-insensitive, ignores extra spaces)
mask_lowq = df["RNA_family"].astype(str).str.strip().str.lower().eq("low quality")
df = df.loc[~mask_lowq].copy()

print("Total usable cells (after mapping + removing low quality):", len(df))
print("RNA_family classes:", df["RNA_family"].nunique())
print(df["RNA_family"].value_counts().head(20))
# ----------------------------
# Drop classes below threshold
# ----------------------------
MIN_SAMPLES_PER_CLASS = 20  # recommended for supervised CNN

class_counts = df["RNA_family"].value_counts()

keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
drop_classes = class_counts[class_counts <  MIN_SAMPLES_PER_CLASS].index.tolist()

print("Keeping classes (>= threshold):", keep_classes)
print("Dropping classes (< threshold):", drop_classes)

df = df[df["RNA_family"].isin(keep_classes)].copy()

print("After thresholding:")
print("  N cells:", len(df))
print("  N classes:", df["RNA_family"].nunique())
print(df["RNA_family"].value_counts())

# ----------------------------
# 3) Balanced split within each RNA_family: 70/15/15
# ----------------------------
rng = np.random.default_rng(RANDOM_SEED)

train_idx, val_idx, test_idx = [], [], []

for fam, g in df.groupby("RNA_family"):
    idx = g.index.to_numpy()
    rng.shuffle(idx)

    n = len(idx)
    n_train = int(np.floor(TRAIN_FRAC * n))
    n_val   = int(np.floor(VAL_FRAC   * n))
    n_test  = n - n_train - n_val  # remainder goes to test

    # (optional safety) if class is tiny, keep at least 1 in train when possible
    if n >= 2 and n_train == 0:
        n_train = 1
        if n_val > 0:
            n_val -= 1
        else:
            n_test -= 1

    tr = idx[:n_train]
    va = idx[n_train:n_train + n_val]
    te = idx[n_train + n_val:]

    train_idx.extend(tr.tolist())
    val_idx.extend(va.tolist())
    test_idx.extend(te.tolist())

df_train = df.loc[train_idx].copy()
df_val   = df.loc[val_idx].copy()
df_test  = df.loc[test_idx].copy()

# ----------------------------
# 4) Report class counts inside each split
# ----------------------------
def report_split(name, d):
    print(f"\n==== {name} ====")
    print("N cells:", len(d))
    print("N classes:", d["RNA_family"].nunique())
    print(d["RNA_family"].value_counts())

report_split("TRAIN", df_train)
report_split("VAL",   df_val)
report_split("TEST",  df_test)

# Crosstab overview (nice sanity check)
ct = pd.DataFrame({
    "train": df_train["RNA_family"].value_counts(),
    "val":   df_val["RNA_family"].value_counts(),
    "test":  df_test["RNA_family"].value_counts(),
}).fillna(0).astype(int)

print("\n==== Per-class counts (train/val/test) ====")
print(ct.sort_values("train", ascending=False))

# ----------------------------
# 5) Save splits
# ----------------------------
# save CSVs
df_train.to_csv(os.path.join(OUTDIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(OUTDIR, "val_split.csv"), index=False)
df_test.to_csv(os.path.join(OUTDIR, "test_split.csv"), index=False)

# also save npy lists of NPZ paths (often most convenient)
np.save(os.path.join(OUTDIR, "train_npz_paths.npy"), df_train["npz_path"].to_numpy())
np.save(os.path.join(OUTDIR, "val_npz_paths.npy"),   df_val["npz_path"].to_numpy())
np.save(os.path.join(OUTDIR, "test_npz_paths.npy"),  df_test["npz_path"].to_numpy())

# save the crosstab
ct.to_csv(os.path.join(OUTDIR, "split_class_counts.csv"))

print("\n✅ Saved splits to:", OUTDIR)


Total usable cells (after mapping + removing low quality): 1172
RNA_family classes: 9
RNA_family
Pvalb    286
Sst      270
IT       220
Vip      152
CT       103
Lamp5     88
ET        35
Sncg      13
NP         5
Name: count, dtype: int64
Keeping classes (>= threshold): ['Pvalb', 'Sst', 'IT', 'Vip', 'CT', 'Lamp5', 'ET']
Dropping classes (< threshold): ['Sncg', 'NP']
After thresholding:
  N cells: 1154
  N classes: 7
RNA_family
Pvalb    286
Sst      270
IT       220
Vip      152
CT       103
Lamp5     88
ET        35
Name: count, dtype: int64

==== TRAIN ====
N cells: 690
N classes: 7
RNA_family
Pvalb    171
Sst      162
IT       132
Vip       91
CT        61
Lamp5     52
ET        21
Name: count, dtype: int64

==== VAL ====
N cells: 113
N classes: 7
RNA_family
Pvalb    28
Sst      27
IT       22
Vip      15
CT       10
Lamp5     8
ET        3
Name: count, dtype: int64

==== TEST ====
N cells: 351
N classes: 7
RNA_family
Pvalb    87
Sst      81
IT       66
Vip      46
CT       32
Lamp5

In [ ]:
# ============================
# THESIS-READY KNN PIPELINE
# - Merge CT/ET/IT
# - Cross-validation
# - Confusion matrices (VAL, TEST, CV)
# - K vs Accuracy curve
# - Balanced accuracy + Macro F1
# - Save all outputs as high-res PNG
# ============================

import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

# ----------------------------
# PATHS
# ----------------------------
META_PATH   = "/content/m1_patchseq_meta_data (1).csv"
SPLIT_DIR   = "/content/dataset_splits_balanced"
FEATURE_CSV = "/content/m1_patchseq_ephys_features.csv"

RESULT_DIR = "./thesis_results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ----------------------------
# PARAMETERS
# ----------------------------
MIN_CLASS_COUNT = 30
K_NEIGHBORS = 25
KNN_METRIC = "cosine"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ----------------------------
# ID HELPERS
# ----------------------------
def any_to_short_id(x):
    x = str(x)
    m = re.search(r"ses-(\d+)-sample-(\d+)", x)
    if m:
        return f"{m.group(1)}_sample_{m.group(2)}"
    m = re.search(r"(\d+)_sample_(\d+)", x)
    if m:
        return f"{m.group(1)}_sample_{m.group(2)}"
    return None

# ----------------------------
# LOAD METADATA + MERGE CLASSES
# ----------------------------
meta = pd.read_csv(META_PATH, sep="\t")
rna_col = [c for c in meta.columns if "rna" in c.lower()][0]

meta["short_id"] = meta["Cell"].apply(any_to_short_id)
meta = meta.dropna(subset=["short_id"]).copy()

meta["RNA_family"] = meta[rna_col].astype(str).str.strip()

# Merge CT/ET/IT
merge_map = {
    "CT": "CT_ET_IT",
    "ET": "CT_ET_IT",
    "IT": "CT_ET_IT",
    "CTandET": "CT_ET_IT",
    "ITandET": "CT_ET_IT"
}
meta["RNA_family"] = meta["RNA_family"].replace(merge_map)

# Remove rare classes
counts = meta["RNA_family"].value_counts()
keep_classes = counts[counts >= MIN_CLASS_COUNT].index
meta = meta[meta["RNA_family"].isin(keep_classes)].copy()

print("Classes used:", keep_classes.tolist())

# ----------------------------
# LOAD FEATURES
# ----------------------------
feat = pd.read_csv(FEATURE_CSV)

best_col = max(feat.columns,
               key=lambda c: feat[c].astype(str).apply(any_to_short_id).notna().sum())

feat["short_id"] = feat[best_col].apply(any_to_short_id)
feat = feat.dropna(subset=["short_id"])

num_cols = [c for c in feat.columns if pd.api.types.is_numeric_dtype(feat[c])]

# ----------------------------
# LOAD SPLITS
# ----------------------------
def load_split(name):
    path = os.path.join(SPLIT_DIR, f"{name}_split.csv")
    df = pd.read_csv(path)
    return df["short_id"].apply(any_to_short_id).dropna().tolist()

train_ids = load_split("train")
val_ids   = load_split("val")
test_ids  = load_split("test")

split_ids = set(train_ids) | set(val_ids) | set(test_ids)

# ----------------------------
# BUILD DATASET
# ----------------------------
meta_ids = set(meta["short_id"])
feat_ids = set(feat["short_id"])

common_ids = split_ids & meta_ids & feat_ids
print("Common cells:", len(common_ids))

df = feat[feat["short_id"].isin(common_ids)].copy()
df["RNA_family"] = df["short_id"].map(
    dict(zip(meta["short_id"], meta["RNA_family"]))
)

df["split"] = df["short_id"].apply(
    lambda x: "train" if x in train_ids else
              "val"   if x in val_ids else
              "test"
)

classes = sorted(df["RNA_family"].unique())
class_to_idx = {c:i for i,c in enumerate(classes)}

def make_xy(d):
    X = d[num_cols].values
    y = d["RNA_family"].map(class_to_idx).values
    return X,y

df_tr = df[df["split"]=="train"]
df_va = df[df["split"]=="val"]
df_te = df[df["split"]=="test"]

Xtr,ytr = make_xy(df_tr)
Xva,yva = make_xy(df_va)
Xte,yte = make_xy(df_te)

# ----------------------------
# TRAIN MODEL
# ----------------------------
knn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=K_NEIGHBORS,
                                  metric=KNN_METRIC))
])
knn.fit(Xtr,ytr)

# ----------------------------
# EVALUATION FUNCTION
# ----------------------------
def evaluate_and_save(X,y,name):
    pred = knn.predict(X)

    acc = accuracy_score(y,pred)
    bal_acc = balanced_accuracy_score(y,pred)
    macro_f1 = f1_score(y,pred,average="macro")

    print(f"\n==== {name} ====")
    print("Accuracy:", round(acc,4))
    print("Balanced Accuracy:", round(bal_acc,4))
    print("Macro F1:", round(macro_f1,4))

    cm = confusion_matrix(y,pred)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt="d",
                xticklabels=classes,
                yticklabels=classes,
                cmap="Blues")
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(f"{RESULT_DIR}/{name}_confusion_matrix.png",dpi=300)
    plt.close()

    # Save classification report
    report = classification_report(y,pred,target_names=classes)
    with open(f"{RESULT_DIR}/{name}_classification_report.txt","w") as f:
        f.write(report)

    return acc

# ----------------------------
# RUN EVALUATION
# ----------------------------
evaluate_and_save(Xva,yva,"VAL")
evaluate_and_save(Xte,yte,"TEST")

# ----------------------------
# STRATIFIED CROSS VALIDATION
# ----------------------------
cv = StratifiedKFold(n_splits=5,shuffle=True,
                     random_state=RANDOM_SEED)

cv_scores = cross_val_score(knn,Xtr,ytr,cv=cv,scoring="accuracy")

print("\nCV Accuracy per fold:",np.round(cv_scores,4))
print("Mean CV:",round(cv_scores.mean(),4))

# CV confusion matrix
y_pred_cv = cross_val_predict(knn,Xtr,ytr,cv=cv)
cm_cv = confusion_matrix(ytr,y_pred_cv)

plt.figure(figsize=(8,6))
sns.heatmap(cm_cv,annot=True,fmt="d",
            xticklabels=classes,
            yticklabels=classes,
            cmap="Greens")
plt.title("Cross-Validated Confusion Matrix (Train)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/CV_confusion_matrix.png",dpi=300)
plt.close()

# ----------------------------
# K vs Accuracy Curve
# ----------------------------
k_values = list(range(1,51,2))
k_scores = []

for k in k_values:
    knn_temp = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k,
                                     metric=KNN_METRIC))
    ])
    scores = cross_val_score(knn_temp,Xtr,ytr,
                             cv=cv,
                             scoring="accuracy")
    k_scores.append(scores.mean())

plt.figure(figsize=(8,5))
plt.plot(k_values,k_scores)
plt.xlabel("K")
plt.ylabel("Cross-Validated Accuracy")
plt.title("K vs CV Accuracy")
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/K_vs_accuracy.png",dpi=300)
plt.close()

print("\nAll thesis figures saved to:",RESULT_DIR)

Classes used: ['CT_ET_IT', 'Pvalb', 'Sst', 'Vip', 'low quality', 'Lamp5']
Common cells: 1154

==== VAL ====
Accuracy: 0.823
Balanced Accuracy: 0.7393
Macro F1: 0.7671

==== TEST ====
Accuracy: 0.849
Balanced Accuracy: 0.7887
Macro F1: 0.808

CV Accuracy per fold: [0.8188 0.8478 0.8623 0.8913 0.7681]
Mean CV: 0.8377

All thesis figures saved to: ./thesis_results


RESNET 18

In [ ]:
import os, re, glob, json, random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

# ============================
# 1) USER SETTINGS & PATHS
# ============================
META_PATH = "/content/m1_patchseq_meta_data (1).csv"  # TAB-delimited
NPZ_DIR   = "/content/WST_kept_cells_sweeps010_070_cut2500_17500_J12_Q20/per_cell_npz"
OUTDIR    = "/content/cross_val_results_Resnet18"
os.makedirs(OUTDIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Hyperparameters
K_FOLDS = 3
BATCH_SIZE = 32
NUM_WORKERS = 2
IMG_SIZE = 224
NUM_EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.25
LABEL_SMOOTH = 0.05
MIXUP_ALPHA = 0.1
EARLY_STOP_PATIENCE = 10
MIN_SAMPLES_PER_CLASS = 20

# ============================
# 2) DATA UTILS & PREPROCESSING
# ============================
# Merge excitatory labels CT/ET/IT into a single label for consistency
MERGE_MAP = {
    "CT": "CT_ET_IT",
    "ET": "CT_ET_IT",
    "IT": "CT_ET_IT",
    "CTandET": "CT_ET_IT",
    "ITandET": "CT_ET_IT",
}

def merge_family_label(lbl: str):
    if lbl is None:
        return None
    lbl = str(lbl).strip()
    return MERGE_MAP.get(lbl, lbl)

def meta_to_short_id(x: str):
    if not isinstance(x, str):
        return None
    m = re.search(r"(\d+)_sample_(\d+)", x.strip())
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

def npz_to_short_id(npz_path: str):
    # Prefer short_id stored inside the NPZ (fast and robust)
    try:
        z = np.load(npz_path, allow_pickle=True)
        if "short_id" in z:
            v = z["short_id"]
            if isinstance(v, np.ndarray):
                v = v[0] if v.size > 0 else ""
            sid = str(v).strip()
            if sid != "":
                return sid
    except Exception:
        pass

    # Fallback: parse from filename
    base = os.path.basename(npz_path)
    m = re.search(r"ses-(\d+)-sample-(\d+)", base)
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

# ============================
# 3) DATASET CLASS
# ============================
class WST_S1_Dataset(Dataset):
    def __init__(self, df, class_to_idx, img_size=224):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        z = np.load(row["npz_path"], allow_pickle=True)

        # Expect S1 exists
        S1 = z["S1"].astype(np.float32)

        # Normalize per-sample
        S1 = (S1 - S1.mean()) / (S1.std() + 1e-6)

        # [H,W] -> [1,1,H,W] -> resize -> repeat to 3 channels -> [3,H,W]
        x = torch.from_numpy(S1)[None, None, :, :]
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        x = x.repeat(1, 3, 1, 1).squeeze(0)

        y = torch.tensor(self.class_to_idx[row["RNA_family"]], dtype=torch.long)
        return x, y

# ============================
# 4) MODEL & TRAINING HELPERS
# ============================
def get_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(DROPOUT_P),
        nn.Linear(in_features, num_classes)
    )
    return model.to(DEVICE)

def mixup_batch(x, y, alpha=0.2):
    if alpha is None or alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # Mixup
        x_mix, y1, y2, lam = mixup_batch(x, y, alpha=MIXUP_ALPHA)

        optimizer.zero_grad()
        logits = model(x_mix)

        # Mixup loss
        if y2 is None:
            loss = criterion(logits, y1)
        else:
            loss = lam * criterion(logits, y1) + (1 - lam) * criterion(logits, y2)

        loss.backward()
        optimizer.step()

        # For reporting: accuracy vs original labels y (approximate under mixup)
        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

    return total_loss / max(total, 1), total_correct / max(total, 1)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    return total_loss / max(total, 1), total_correct / max(total, 1), all_labels, all_preds

def save_confusion_matrix_png(cm, class_names, out_png, title):
    # Thesis-friendly: readable sizing + 300 dpi
    plt.figure(figsize=(1.2 * len(class_names), 1.0 * len(class_names)))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    # annotate
    thresh = cm.max() * 0.5 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_loss_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_acc_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# ============================
# 5) LOAD METADATA + NPZ INDEX
# ============================
print("\n--- Loading metadata ---")
meta = pd.read_csv(META_PATH, sep="\t")

# find RNA family column robustly
rna_cols = [c for c in meta.columns if c.strip().lower().replace(" ", "") == "rnafamily"]
if len(rna_cols) == 0:
    raise ValueError("Could not find a column named 'RNA family' (RNAfamily) in metadata.")
rna_col = rna_cols[0]

meta_short = meta["Cell"].astype(str).apply(meta_to_short_id)
rna_raw = meta[rna_col].astype(str)
rna_map = dict(zip(meta_short, rna_raw))

print("--- Indexing NPZ files ---")
npz_paths = sorted(glob.glob(os.path.join(NPZ_DIR, "*.npz")))

rows = []
for p in npz_paths:
    sid = npz_to_short_id(p)
    fam_raw = rna_map.get(sid, None)
    fam = merge_family_label(fam_raw)
    if sid is None or fam is None:
        continue
    if str(fam).strip().lower() == "low quality":
        continue
    rows.append((p, sid, fam))

df = pd.DataFrame(rows, columns=["npz_path", "short_id", "RNA_family"])

if len(df) == 0:
    raise ValueError("No usable samples found after mapping NPZs to metadata labels.")

# filter rare classes
counts = df["RNA_family"].value_counts()
keep_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
drop_classes = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()

df = df[df["RNA_family"].isin(keep_classes)].copy()

classes = sorted(df["RNA_family"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

print("\n--- Dataset summary (after merge + filtering) ---")
print(f"Total samples: {len(df)}")
print(f"Classes kept ({len(classes)}): {classes}")
print("\nClass counts:")
print(df["RNA_family"].value_counts().to_string())
if len(drop_classes) > 0:
    print("\nDropped classes (< MIN_SAMPLES_PER_CLASS):", drop_classes)

# ============================
# 6) HOLD-OUT FINAL TEST SET
# ============================
df_cv, df_test = train_test_split(
    df, test_size=0.2, stratify=df["RNA_family"], random_state=SEED
)

print("\n--- Split summary ---")
print(f"CV pool:   {len(df_cv)}")
print(f"Test set:  {len(df_test)}")

# Save these splits for reproducibility
df_cv.to_csv(os.path.join(OUTDIR, "cv_pool.csv"), index=False)
df_test.to_csv(os.path.join(OUTDIR, "final_test_set.csv"), index=False)

# ============================
# 7) K-FOLD CROSS VALIDATION
# ============================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_best_acc = []
fold_model_paths = []
all_fold_summaries = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df_cv, df_cv["RNA_family"]), start=1):
    print(f"\n==============================")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"==============================")

    fold_dir = os.path.join(OUTDIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    train_sub = df_cv.iloc[train_idx].copy()
    val_sub   = df_cv.iloc[val_idx].copy()

    # Compute class weights (avoid division by zero)
    train_counts = train_sub["RNA_family"].value_counts().reindex(classes).fillna(0).astype(int).values
    safe_counts = np.maximum(train_counts, 1)  # avoid zero
    weights = (len(train_sub) / (len(classes) * safe_counts)).astype(np.float32)
    weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

    # Datasets / loaders
    train_ds = WST_S1_Dataset(train_sub, class_to_idx, img_size=IMG_SIZE)
    val_ds   = WST_S1_Dataset(val_sub, class_to_idx, img_size=IMG_SIZE)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    # Model / optimizer / loss / scheduler
    model = get_model(len(classes))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_val_acc = -1.0
    best_epoch = -1
    patience_counter = 0

    history = []  # store per-epoch metrics for thesis

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, y_val_true, y_val_pred = evaluate(model, val_loader, criterion)

        scheduler.step(val_acc)

        history.append({
            "epoch": epoch,
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        improved = val_acc > best_val_acc + 1e-6
        if improved:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            best_model_path = os.path.join(fold_dir, "best_model.pth")
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1

        # Cleaner prints every epoch (thesis logging is saved anyway)
        print(f"Epoch {epoch:02d} | "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} (best epoch = {best_epoch}, best val acc = {best_val_acc:.4f})")
            break

    # Save history (CSV + JSON)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(fold_dir, "history.csv"), index=False)
    with open(os.path.join(fold_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Save curves as PNG
    save_loss_curves_png(hist_df, os.path.join(fold_dir, "loss_curve.png"),
                         title=f"Fold {fold} Loss Curve")
    save_acc_curves_png(hist_df, os.path.join(fold_dir, "acc_curve.png"),
                        title=f"Fold {fold} Accuracy Curve")

    # Load best model for fold and compute fold confusion matrix on val
    best_model = get_model(len(classes))
    best_model.load_state_dict(torch.load(os.path.join(fold_dir, "best_model.pth"), map_location=DEVICE))
    best_model.eval()

    val_loss_b, val_acc_b, y_val_true_b, y_val_pred_b = evaluate(best_model, val_loader, nn.CrossEntropyLoss())

    cm_val = confusion_matrix(y_val_true_b, y_val_pred_b, labels=list(range(len(classes))))
    cm_csv = os.path.join(fold_dir, "val_confusion_matrix.csv")
    pd.DataFrame(cm_val, index=classes, columns=classes).to_csv(cm_csv)

    cm_png = os.path.join(fold_dir, "val_confusion_matrix.png")
    save_confusion_matrix_png(cm_val, classes, cm_png, title=f"Fold {fold} VAL Confusion Matrix (acc={val_acc_b:.3f})")

    # Save classification report (VAL)
    report = classification_report(
        y_val_true_b, y_val_pred_b,
        labels=list(range(len(classes))),
        target_names=classes,
        digits=4,
        zero_division=0
    )
    with open(os.path.join(fold_dir, "val_classification_report.txt"), "w") as f:
        f.write(report)

    # Store fold summary
    fold_best_acc.append(float(best_val_acc))
    fold_model_paths.append(os.path.join(fold_dir, "best_model.pth"))
    all_fold_summaries.append({
        "fold": fold,
        "best_val_acc": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "n_train": int(len(train_sub)),
        "n_val": int(len(val_sub)),
    })

    print(f"\nFold {fold} summary:")
    print(f"  best_val_acc = {best_val_acc:.4f} at epoch {best_epoch}")
    print(f"  saved: {fold_dir}/best_model.pth")
    print(f"  saved: {fold_dir}/loss_curve.png, {fold_dir}/acc_curve.png, {fold_dir}/val_confusion_matrix.png")

# Save CV summary for thesis
cv_mean = float(np.mean(fold_best_acc))
cv_std  = float(np.std(fold_best_acc))

summary = {
    "k_folds": K_FOLDS,
    "fold_best_val_acc": fold_best_acc,
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "classes": classes,
    "n_total_samples": int(len(df)),
    "n_cv_pool": int(len(df_cv)),
    "n_final_test": int(len(df_test)),
    "hyperparams": {
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "epochs_max": NUM_EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout_p": DROPOUT_P,
        "label_smoothing": LABEL_SMOOTH,
        "mixup_alpha": MIXUP_ALPHA,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS
    },
    "fold_details": all_fold_summaries
}
with open(os.path.join(OUTDIR, "cv_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n==============================")
print("CROSS-VALIDATION SUMMARY")
print("==============================")
print(f"Fold best val acc: {np.round(fold_best_acc, 4).tolist()}")
print(f"CV mean(best val acc) = {cv_mean:.4f} ± {cv_std:.4f}")
print(f"Saved CV summary to: {os.path.join(OUTDIR, 'cv_summary.json')}")

# ============================
# 8) FINAL EVALUATION ON UNSEEN TEST SET
# ============================
best_fold_idx = int(np.argmax(fold_best_acc))  # 0-based index
best_model_path = fold_model_paths[best_fold_idx]

print("\n==============================")
print("FINAL TEST EVALUATION")
print("==============================")
print(f"Best fold = {best_fold_idx + 1} with best_val_acc = {fold_best_acc[best_fold_idx]:.4f}")
print(f"Loading model from: {best_model_path}")

# Build a fresh model for test eval (important)
final_model = get_model(len(classes))
final_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
final_model.eval()

test_ds = WST_S1_Dataset(df_test, class_to_idx, img_size=IMG_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

test_criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(final_model, test_loader, test_criterion)

print(f"Test loss = {test_loss:.4f}")
print(f"Test acc  = {test_acc:.4f}")

# Save test report
test_report = classification_report(
    y_true, y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=4,
    zero_division=0
)
with open(os.path.join(OUTDIR, "final_test_classification_report.txt"), "w") as f:
    f.write(test_report)

# Confusion matrix (test)
cm_test = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
pd.DataFrame(cm_test, index=classes, columns=classes).to_csv(os.path.join(OUTDIR, "final_test_confusion_matrix.csv"))

save_confusion_matrix_png(
    cm_test, classes,
    os.path.join(OUTDIR, "final_test_confusion_matrix.png"),
    title=f"FINAL TEST Confusion Matrix (acc={test_acc:.3f})"
)

# Save a small thesis-friendly metrics file
final_metrics = {
    "best_fold": best_fold_idx + 1,
    "best_fold_val_acc": float(fold_best_acc[best_fold_idx]),
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "final_test_loss": float(test_loss),
    "final_test_acc": float(test_acc),
}
with open(os.path.join(OUTDIR, "final_metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=2)

print("\nSaved thesis-ready outputs to:", OUTDIR)
print("Files you will use in thesis (PNG):")
print("  - fold_*/loss_curve.png")
print("  - fold_*/acc_curve.png")
print("  - fold_*/val_confusion_matrix.png")
print("  - final_test_confusion_matrix.png")
print("Plus CSV/TXT/JSON reports for appendix.")


--- Loading metadata ---
--- Indexing NPZ files ---

--- Dataset summary (after merge + filtering) ---
Total samples: 1154
Classes kept (5): ['CT_ET_IT', 'Lamp5', 'Pvalb', 'Sst', 'Vip']

Class counts:
RNA_family
CT_ET_IT    358
Pvalb       286
Sst         270
Vip         152
Lamp5        88

Dropped classes (< MIN_SAMPLES_PER_CLASS): ['Sncg', 'NP']

--- Split summary ---
CV pool:   923
Test set:  231

FOLD 1/3
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


Epoch 01 | train_loss=0.9544 train_acc=0.463 | val_loss=0.8317 val_acc=0.782 | lr=3.00e-04
Epoch 02 | train_loss=0.6494 train_acc=0.481 | val_loss=0.8306 val_acc=0.734 | lr=3.00e-04
Epoch 03 | train_loss=0.5982 train_acc=0.676 | val_loss=0.9470 val_acc=0.795 | lr=3.00e-04
Epoch 04 | train_loss=0.5231 train_acc=0.571 | val_loss=0.6429 val_acc=0.893 | lr=3.00e-04
Epoch 05 | train_loss=0.5131 train_acc=0.701 | val_loss=0.5947 val_acc=0.903 | lr=3.00e-04
Epoch 06 | train_loss=0.5150 train_acc=0.657 | val_loss=0.7537 val_acc=0.828 | lr=3.00e-04
Epoch 07 | train_loss=0.6219 train_acc=0.629 | val_loss=0.5849 val_acc=0.906 | lr=3.00e-04
Epoch 08 | train_loss=0.4578 train_acc=0.623 | val_loss=0.5823 val_acc=0.919 | lr=3.00e-04
Epoch 09 | train_loss=0.4893 train_acc=0.626 | val_loss=0.5904 val_acc=0.925 | lr=3.00e-04
Epoch 10 | train_loss=0.4566 train_acc=0.593 | val_loss=0.5573 val_acc=0.919 | lr=3.00e-04
Epoch 11 | train_loss=0.4389 train_acc=0.559 | val_loss=0.5610 val_acc=0.909 | lr=3.00e-04

VGG 16

In [ ]:
import os, re, glob, json, random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

# ============================
# 1) USER SETTINGS & PATHS
# ============================
META_PATH = "/content/m1_patchseq_meta_data (1).csv"  # TAB-delimited
NPZ_DIR   = "/content/WST_kept_cells_sweeps010_070_cut2500_17500_J12_Q20/per_cell_npz"
OUTDIR    = "/content/cross_val_results_VGG16"
os.makedirs(OUTDIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Hyperparameters
K_FOLDS = 3
BATCH_SIZE = 32
NUM_WORKERS = 2
IMG_SIZE = 224
NUM_EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.25
LABEL_SMOOTH = 0.05
MIXUP_ALPHA = 0.1
EARLY_STOP_PATIENCE = 10
MIN_SAMPLES_PER_CLASS = 20

# ============================
# 2) DATA UTILS & PREPROCESSING
# ============================
# Merge excitatory labels CT/ET/IT into a single label for consistency
MERGE_MAP = {
    "CT": "CT_ET_IT",
    "ET": "CT_ET_IT",
    "IT": "CT_ET_IT",
    "CTandET": "CT_ET_IT",
    "ITandET": "CT_ET_IT",
}

def merge_family_label(lbl: str):
    if lbl is None:
        return None
    lbl = str(lbl).strip()
    return MERGE_MAP.get(lbl, lbl)

def meta_to_short_id(x: str):
    if not isinstance(x, str):
        return None
    m = re.search(r"(\d+)_sample_(\d+)", x.strip())
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

def npz_to_short_id(npz_path: str):
    # Prefer short_id stored inside the NPZ (fast and robust)
    try:
        z = np.load(npz_path, allow_pickle=True)
        if "short_id" in z:
            v = z["short_id"]
            if isinstance(v, np.ndarray):
                v = v[0] if v.size > 0 else ""
            sid = str(v).strip()
            if sid != "":
                return sid
    except Exception:
        pass

    # Fallback: parse from filename
    base = os.path.basename(npz_path)
    m = re.search(r"ses-(\d+)-sample-(\d+)", base)
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

# ============================
# 3) DATASET CLASS
# ============================
class WST_S1_Dataset(Dataset):
    def __init__(self, df, class_to_idx, img_size=224):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        z = np.load(row["npz_path"], allow_pickle=True)

        # Expect S1 exists
        S1 = z["S1"].astype(np.float32)

        # Normalize per-sample
        S1 = (S1 - S1.mean()) / (S1.std() + 1e-6)

        # [H,W] -> [1,1,H,W] -> resize -> repeat to 3 channels -> [3,H,W]
        x = torch.from_numpy(S1)[None, None, :, :]
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        x = x.repeat(1, 3, 1, 1).squeeze(0)

        y = torch.tensor(self.class_to_idx[row["RNA_family"]], dtype=torch.long)
        return x, y

# ============================
# 4) MODEL & TRAINING HELPERS
# ============================
def get_model(n_classes):
    model = models.vgg16(
        weights=models.VGG16_Weights.DEFAULT
    )
    in_f = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_f, n_classes)
    return model.to(DEVICE)


def mixup_batch(x, y, alpha=0.2):
    if alpha is None or alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # Mixup
        x_mix, y1, y2, lam = mixup_batch(x, y, alpha=MIXUP_ALPHA)

        optimizer.zero_grad()
        logits = model(x_mix)

        # Mixup loss
        if y2 is None:
            loss = criterion(logits, y1)
        else:
            loss = lam * criterion(logits, y1) + (1 - lam) * criterion(logits, y2)

        loss.backward()
        optimizer.step()

        # For reporting: accuracy vs original labels y (approximate under mixup)
        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

    return total_loss / max(total, 1), total_correct / max(total, 1)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    return total_loss / max(total, 1), total_correct / max(total, 1), all_labels, all_preds

def save_confusion_matrix_png(cm, class_names, out_png, title):
    # Thesis-friendly: readable sizing + 300 dpi
    plt.figure(figsize=(1.2 * len(class_names), 1.0 * len(class_names)))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    # annotate
    thresh = cm.max() * 0.5 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_loss_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_acc_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# ============================
# 5) LOAD METADATA + NPZ INDEX
# ============================
print("\n--- Loading metadata ---")
meta = pd.read_csv(META_PATH, sep="\t")

# find RNA family column robustly
rna_cols = [c for c in meta.columns if c.strip().lower().replace(" ", "") == "rnafamily"]
if len(rna_cols) == 0:
    raise ValueError("Could not find a column named 'RNA family' (RNAfamily) in metadata.")
rna_col = rna_cols[0]

meta_short = meta["Cell"].astype(str).apply(meta_to_short_id)
rna_raw = meta[rna_col].astype(str)
rna_map = dict(zip(meta_short, rna_raw))

print("--- Indexing NPZ files ---")
npz_paths = sorted(glob.glob(os.path.join(NPZ_DIR, "*.npz")))

rows = []
for p in npz_paths:
    sid = npz_to_short_id(p)
    fam_raw = rna_map.get(sid, None)
    fam = merge_family_label(fam_raw)
    if sid is None or fam is None:
        continue
    if str(fam).strip().lower() == "low quality":
        continue
    rows.append((p, sid, fam))

df = pd.DataFrame(rows, columns=["npz_path", "short_id", "RNA_family"])

if len(df) == 0:
    raise ValueError("No usable samples found after mapping NPZs to metadata labels.")

# filter rare classes
counts = df["RNA_family"].value_counts()
keep_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
drop_classes = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()

df = df[df["RNA_family"].isin(keep_classes)].copy()

classes = sorted(df["RNA_family"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

print("\n--- Dataset summary (after merge + filtering) ---")
print(f"Total samples: {len(df)}")
print(f"Classes kept ({len(classes)}): {classes}")
print("\nClass counts:")
print(df["RNA_family"].value_counts().to_string())
if len(drop_classes) > 0:
    print("\nDropped classes (< MIN_SAMPLES_PER_CLASS):", drop_classes)

# ============================
# 6) HOLD-OUT FINAL TEST SET
# ============================
df_cv, df_test = train_test_split(
    df, test_size=0.2, stratify=df["RNA_family"], random_state=SEED
)

print("\n--- Split summary ---")
print(f"CV pool:   {len(df_cv)}")
print(f"Test set:  {len(df_test)}")

# Save these splits for reproducibility
df_cv.to_csv(os.path.join(OUTDIR, "cv_pool.csv"), index=False)
df_test.to_csv(os.path.join(OUTDIR, "final_test_set.csv"), index=False)

# ============================
# 7) K-FOLD CROSS VALIDATION
# ============================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_best_acc = []
fold_model_paths = []
all_fold_summaries = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df_cv, df_cv["RNA_family"]), start=1):
    print(f"\n==============================")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"==============================")

    fold_dir = os.path.join(OUTDIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    train_sub = df_cv.iloc[train_idx].copy()
    val_sub   = df_cv.iloc[val_idx].copy()

    # Compute class weights (avoid division by zero)
    train_counts = train_sub["RNA_family"].value_counts().reindex(classes).fillna(0).astype(int).values
    safe_counts = np.maximum(train_counts, 1)  # avoid zero
    weights = (len(train_sub) / (len(classes) * safe_counts)).astype(np.float32)
    weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

    # Datasets / loaders
    train_ds = WST_S1_Dataset(train_sub, class_to_idx, img_size=IMG_SIZE)
    val_ds   = WST_S1_Dataset(val_sub, class_to_idx, img_size=IMG_SIZE)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    # Model / optimizer / loss / scheduler
    model = get_model(len(classes))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_val_acc = -1.0
    best_epoch = -1
    patience_counter = 0

    history = []  # store per-epoch metrics for thesis

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, y_val_true, y_val_pred = evaluate(model, val_loader, criterion)

        scheduler.step(val_acc)

        history.append({
            "epoch": epoch,
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        improved = val_acc > best_val_acc + 1e-6
        if improved:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            best_model_path = os.path.join(fold_dir, "best_model.pth")
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1

        # Cleaner prints every epoch (thesis logging is saved anyway)
        print(f"Epoch {epoch:02d} | "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} (best epoch = {best_epoch}, best val acc = {best_val_acc:.4f})")
            break

    # Save history (CSV + JSON)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(fold_dir, "history.csv"), index=False)
    with open(os.path.join(fold_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Save curves as PNG
    save_loss_curves_png(hist_df, os.path.join(fold_dir, "loss_curve.png"),
                         title=f"Fold {fold} Loss Curve")
    save_acc_curves_png(hist_df, os.path.join(fold_dir, "acc_curve.png"),
                        title=f"Fold {fold} Accuracy Curve")

    # Load best model for fold and compute fold confusion matrix on val
    best_model = get_model(len(classes))
    best_model.load_state_dict(torch.load(os.path.join(fold_dir, "best_model.pth"), map_location=DEVICE))
    best_model.eval()

    val_loss_b, val_acc_b, y_val_true_b, y_val_pred_b = evaluate(best_model, val_loader, nn.CrossEntropyLoss())

    cm_val = confusion_matrix(y_val_true_b, y_val_pred_b, labels=list(range(len(classes))))
    cm_csv = os.path.join(fold_dir, "val_confusion_matrix.csv")
    pd.DataFrame(cm_val, index=classes, columns=classes).to_csv(cm_csv)

    cm_png = os.path.join(fold_dir, "val_confusion_matrix.png")
    save_confusion_matrix_png(cm_val, classes, cm_png, title=f"Fold {fold} VAL Confusion Matrix (acc={val_acc_b:.3f})")

    # Save classification report (VAL)
    report = classification_report(
        y_val_true_b, y_val_pred_b,
        labels=list(range(len(classes))),
        target_names=classes,
        digits=4,
        zero_division=0
    )
    with open(os.path.join(fold_dir, "val_classification_report.txt"), "w") as f:
        f.write(report)

    # Store fold summary
    fold_best_acc.append(float(best_val_acc))
    fold_model_paths.append(os.path.join(fold_dir, "best_model.pth"))
    all_fold_summaries.append({
        "fold": fold,
        "best_val_acc": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "n_train": int(len(train_sub)),
        "n_val": int(len(val_sub)),
    })

    print(f"\nFold {fold} summary:")
    print(f"  best_val_acc = {best_val_acc:.4f} at epoch {best_epoch}")
    print(f"  saved: {fold_dir}/best_model.pth")
    print(f"  saved: {fold_dir}/loss_curve.png, {fold_dir}/acc_curve.png, {fold_dir}/val_confusion_matrix.png")

# Save CV summary for thesis
cv_mean = float(np.mean(fold_best_acc))
cv_std  = float(np.std(fold_best_acc))

summary = {
    "k_folds": K_FOLDS,
    "fold_best_val_acc": fold_best_acc,
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "classes": classes,
    "n_total_samples": int(len(df)),
    "n_cv_pool": int(len(df_cv)),
    "n_final_test": int(len(df_test)),
    "hyperparams": {
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "epochs_max": NUM_EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout_p": DROPOUT_P,
        "label_smoothing": LABEL_SMOOTH,
        "mixup_alpha": MIXUP_ALPHA,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS
    },
    "fold_details": all_fold_summaries
}
with open(os.path.join(OUTDIR, "cv_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n==============================")
print("CROSS-VALIDATION SUMMARY")
print("==============================")
print(f"Fold best val acc: {np.round(fold_best_acc, 4).tolist()}")
print(f"CV mean(best val acc) = {cv_mean:.4f} ± {cv_std:.4f}")
print(f"Saved CV summary to: {os.path.join(OUTDIR, 'cv_summary.json')}")

# ============================
# 8) FINAL EVALUATION ON UNSEEN TEST SET
# ============================
best_fold_idx = int(np.argmax(fold_best_acc))  # 0-based index
best_model_path = fold_model_paths[best_fold_idx]

print("\n==============================")
print("FINAL TEST EVALUATION")
print("==============================")
print(f"Best fold = {best_fold_idx + 1} with best_val_acc = {fold_best_acc[best_fold_idx]:.4f}")
print(f"Loading model from: {best_model_path}")

# Build a fresh model for test eval (important)
final_model = get_model(len(classes))
final_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
final_model.eval()

test_ds = WST_S1_Dataset(df_test, class_to_idx, img_size=IMG_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

test_criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(final_model, test_loader, test_criterion)

print(f"Test loss = {test_loss:.4f}")
print(f"Test acc  = {test_acc:.4f}")

# Save test report
test_report = classification_report(
    y_true, y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=4,
    zero_division=0
)
with open(os.path.join(OUTDIR, "final_test_classification_report.txt"), "w") as f:
    f.write(test_report)

# Confusion matrix (test)
cm_test = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
pd.DataFrame(cm_test, index=classes, columns=classes).to_csv(os.path.join(OUTDIR, "final_test_confusion_matrix.csv"))

save_confusion_matrix_png(
    cm_test, classes,
    os.path.join(OUTDIR, "final_test_confusion_matrix.png"),
    title=f"FINAL TEST Confusion Matrix (acc={test_acc:.3f})"
)

# Save a small thesis-friendly metrics file
final_metrics = {
    "best_fold": best_fold_idx + 1,
    "best_fold_val_acc": float(fold_best_acc[best_fold_idx]),
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "final_test_loss": float(test_loss),
    "final_test_acc": float(test_acc),
}
with open(os.path.join(OUTDIR, "final_metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=2)

print("\nSaved thesis-ready outputs to:", OUTDIR)
print("Files you will use in thesis (PNG):")
print("  - fold_*/loss_curve.png")
print("  - fold_*/acc_curve.png")
print("  - fold_*/val_confusion_matrix.png")
print("  - final_test_confusion_matrix.png")
print("Plus CSV/TXT/JSON reports for appendix.")


--- Loading metadata ---
--- Indexing NPZ files ---

--- Dataset summary (after merge + filtering) ---
Total samples: 1154
Classes kept (5): ['CT_ET_IT', 'Lamp5', 'Pvalb', 'Sst', 'Vip']

Class counts:
RNA_family
CT_ET_IT    358
Pvalb       286
Sst         270
Vip         152
Lamp5        88

Dropped classes (< MIN_SAMPLES_PER_CLASS): ['Sncg', 'NP']

--- Split summary ---
CV pool:   923
Test set:  231

FOLD 1/3
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 138MB/s]


Epoch 01 | train_loss=1.4491 train_acc=0.384 | val_loss=1.1707 val_acc=0.510 | lr=3.00e-04
Epoch 02 | train_loss=1.0274 train_acc=0.410 | val_loss=0.9232 val_acc=0.636 | lr=3.00e-04
Epoch 03 | train_loss=0.9982 train_acc=0.522 | val_loss=0.7531 val_acc=0.779 | lr=3.00e-04
Epoch 04 | train_loss=0.9979 train_acc=0.415 | val_loss=0.7921 val_acc=0.844 | lr=3.00e-04
Epoch 05 | train_loss=0.8336 train_acc=0.603 | val_loss=0.7856 val_acc=0.841 | lr=3.00e-04
Epoch 06 | train_loss=0.8154 train_acc=0.600 | val_loss=0.8618 val_acc=0.711 | lr=3.00e-04
Epoch 07 | train_loss=0.8637 train_acc=0.554 | val_loss=0.6299 val_acc=0.838 | lr=1.50e-04
Epoch 08 | train_loss=0.5640 train_acc=0.592 | val_loss=0.6345 val_acc=0.919 | lr=1.50e-04
Epoch 09 | train_loss=0.5849 train_acc=0.590 | val_loss=0.5905 val_acc=0.883 | lr=1.50e-04
Epoch 10 | train_loss=0.5287 train_acc=0.572 | val_loss=0.5898 val_acc=0.873 | lr=1.50e-04
Epoch 11 | train_loss=0.4869 train_acc=0.550 | val_loss=0.5325 val_acc=0.896 | lr=7.50e-05

INCEPTION V3

In [ ]:
# ============================================================
# COMPLETE, FIXED SCRIPT (Inception v3) — NO "5x5 > 3x3" ERROR
# Key fixes:
#   1) Inception expects 299x299  (IMG_SIZE=299)
#   2) aux_logits=False
#   3) Safe forward() in case torchvision returns tuple
# ============================================================

import os, re, glob, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

# ============================
# 1) USER SETTINGS & PATHS
# ============================
META_PATH = "/content/m1_patchseq_meta_data (1).csv"  # TAB-delimited
NPZ_DIR   = "/content/WST_kept_cells_sweeps010_070_cut2500_17500_J12_Q20/per_cell_npz"
OUTDIR    = "/content/cross_val_results_inception"
os.makedirs(OUTDIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Hyperparameters
K_FOLDS = 3
BATCH_SIZE = 32
NUM_WORKERS = 2

# IMPORTANT FOR INCEPTION V3
IMG_SIZE = 299

NUM_EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
MIXUP_ALPHA = 0.1
EARLY_STOP_PATIENCE = 10
MIN_SAMPLES_PER_CLASS = 20

# ============================
# 2) DATA UTILS & PREPROCESSING
# ============================
# Merge excitatory labels CT/ET/IT into a single label for consistency
MERGE_MAP = {
    "CT": "CT_ET_IT",
    "ET": "CT_ET_IT",
    "IT": "CT_ET_IT",
    "CTandET": "CT_ET_IT",
    "ITandET": "CT_ET_IT",
}

def merge_family_label(lbl: str):
    if lbl is None:
        return None
    lbl = str(lbl).strip()
    return MERGE_MAP.get(lbl, lbl)

def meta_to_short_id(x: str):
    if not isinstance(x, str):
        return None
    m = re.search(r"(\d+)_sample_(\d+)", x.strip())
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

def npz_to_short_id(npz_path: str):
    # Prefer short_id stored inside the NPZ (fast and robust)
    try:
        z = np.load(npz_path, allow_pickle=True)
        if "short_id" in z:
            v = z["short_id"]
            if isinstance(v, np.ndarray):
                v = v[0] if v.size > 0 else ""
            sid = str(v).strip()
            if sid != "":
                return sid
    except Exception:
        pass

    # Fallback: parse from filename
    base = os.path.basename(npz_path)
    m = re.search(r"ses-(\d+)-sample-(\d+)", base)
    return f"{m.group(1)}_sample_{m.group(2)}" if m else None

# ============================
# 3) DATASET CLASS
# ============================
class WST_S1_Dataset(Dataset):
    def __init__(self, df, class_to_idx, img_size=299):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        z = np.load(row["npz_path"], allow_pickle=True)

        # Expect S1 exists
        S1 = z["S1"].astype(np.float32)

        # Normalize per-sample
        S1 = (S1 - S1.mean()) / (S1.std() + 1e-6)

        # [H,W] -> [1,1,H,W] -> resize -> repeat to 3 channels -> [3,H,W]
        x = torch.from_numpy(S1)[None, None, :, :]
        x = F.interpolate(
            x, size=(self.img_size, self.img_size),
            mode="bilinear", align_corners=False
        )
        x = x.repeat(1, 3, 1, 1).squeeze(0)

        y = torch.tensor(self.class_to_idx[row["RNA_family"]], dtype=torch.long)
        return x, y

# ============================
# MODEL (SAFE VERSION)
# ============================

def get_model(n_classes):

    model = models.inception_v3(
        weights=models.Inception_V3_Weights.DEFAULT
        # DO NOT set aux_logits manually
    )

    # Replace final classifier
    in_f = model.fc.in_features
    model.fc = nn.Linear(in_f, n_classes)

    return model.to(DEVICE)

def forward_logits(model, x):
    out = model(x)

    # torchvision inception returns tuple in training mode
    if isinstance(out, tuple):
        out = out[0]  # ignore aux output

    return out

# ============================
# TRAIN / EVAL
# ============================

def mixup_batch(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        x_mix, y1, y2, lam = mixup_batch(x, y, MIXUP_ALPHA)

        optimizer.zero_grad()
        logits = forward_logits(model, x_mix)

        if y2 is None:
            loss = criterion(logits, y1)
        else:
            loss = lam * criterion(logits, y1) + (1 - lam) * criterion(logits, y2)

        loss.backward()
        optimizer.step()

        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = forward_logits(model, x)
        loss = criterion(logits, y)

        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    return total_loss / total, total_correct / total, all_labels, all_preds

def save_confusion_matrix_png(cm, class_names, out_png, title):
    plt.figure(figsize=(1.2 * len(class_names), 1.0 * len(class_names)))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() * 0.5 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_loss_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

def save_acc_curves_png(history_df, out_png, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# ============================
# 5) LOAD METADATA + NPZ INDEX
# ============================
print("\n--- Loading metadata ---")
meta = pd.read_csv(META_PATH, sep="\t")

# find RNA family column robustly
rna_cols = [c for c in meta.columns if c.strip().lower().replace(" ", "") == "rnafamily"]
if len(rna_cols) == 0:
    raise ValueError("Could not find a column named 'RNA family' (RNAfamily) in metadata.")
rna_col = rna_cols[0]

meta_short = meta["Cell"].astype(str).apply(meta_to_short_id)
rna_raw = meta[rna_col].astype(str)
rna_map = dict(zip(meta_short, rna_raw))

print("--- Indexing NPZ files ---")
npz_paths = sorted(glob.glob(os.path.join(NPZ_DIR, "*.npz")))

rows = []
for p in npz_paths:
    sid = npz_to_short_id(p)
    fam_raw = rna_map.get(sid, None)
    fam = merge_family_label(fam_raw)
    if sid is None or fam is None:
        continue
    if str(fam).strip().lower() == "low quality":
        continue
    rows.append((p, sid, fam))

df = pd.DataFrame(rows, columns=["npz_path", "short_id", "RNA_family"])
if len(df) == 0:
    raise ValueError("No usable samples found after mapping NPZs to metadata labels.")

# filter rare classes
counts = df["RNA_family"].value_counts()
keep_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
drop_classes = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()
df = df[df["RNA_family"].isin(keep_classes)].copy()

classes = sorted(df["RNA_family"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

print("\n--- Dataset summary (after merge + filtering) ---")
print(f"Total samples: {len(df)}")
print(f"Classes kept ({len(classes)}): {classes}")
print("\nClass counts:")
print(df["RNA_family"].value_counts().to_string())
if len(drop_classes) > 0:
    print("\nDropped classes (< MIN_SAMPLES_PER_CLASS):", drop_classes)

# ============================
# 6) HOLD-OUT FINAL TEST SET
# ============================
df_cv, df_test = train_test_split(
    df, test_size=0.2, stratify=df["RNA_family"], random_state=SEED
)

print("\n--- Split summary ---")
print(f"CV pool:   {len(df_cv)}")
print(f"Test set:  {len(df_test)}")

df_cv.to_csv(os.path.join(OUTDIR, "cv_pool.csv"), index=False)
df_test.to_csv(os.path.join(OUTDIR, "final_test_set.csv"), index=False)

# ============================
# 7) K-FOLD CROSS VALIDATION
# ============================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_best_acc = []
fold_model_paths = []
all_fold_summaries = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df_cv, df_cv["RNA_family"]), start=1):
    print(f"\n==============================")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"==============================")

    fold_dir = os.path.join(OUTDIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    train_sub = df_cv.iloc[train_idx].copy()
    val_sub   = df_cv.iloc[val_idx].copy()

    # class weights based on train fold only
    train_counts = train_sub["RNA_family"].value_counts().reindex(classes).fillna(0).astype(int).values
    safe_counts = np.maximum(train_counts, 1)
    weights = (len(train_sub) / (len(classes) * safe_counts)).astype(np.float32)
    weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

    train_ds = WST_S1_Dataset(train_sub, class_to_idx, img_size=IMG_SIZE)
    val_ds   = WST_S1_Dataset(val_sub, class_to_idx, img_size=IMG_SIZE)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    model = get_model(len(classes))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_val_acc = -1.0
    best_epoch = -1
    patience_counter = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, y_val_true, y_val_pred = evaluate(model, val_loader, criterion)

        scheduler.step(val_acc)

        history.append({
            "epoch": epoch,
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        improved = val_acc > best_val_acc + 1e-6
        if improved:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            best_model_path = os.path.join(fold_dir, "best_model.pth")
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(
                f"Early stopping at epoch {epoch} "
                f"(best epoch={best_epoch}, best val acc={best_val_acc:.4f})"
            )
            break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(fold_dir, "history.csv"), index=False)
    with open(os.path.join(fold_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    save_loss_curves_png(hist_df, os.path.join(fold_dir, "loss_curve.png"),
                         title=f"Fold {fold} Loss Curve")
    save_acc_curves_png(hist_df, os.path.join(fold_dir, "acc_curve.png"),
                        title=f"Fold {fold} Accuracy Curve")

    # Evaluate best model for fold on validation (no weights/smoothing for reporting)
    best_model = get_model(len(classes))
    best_model.load_state_dict(torch.load(os.path.join(fold_dir, "best_model.pth"), map_location=DEVICE))
    best_model.eval()

    val_loss_b, val_acc_b, y_val_true_b, y_val_pred_b = evaluate(
        best_model, val_loader, nn.CrossEntropyLoss()
    )

    cm_val = confusion_matrix(y_val_true_b, y_val_pred_b, labels=list(range(len(classes))))
    pd.DataFrame(cm_val, index=classes, columns=classes).to_csv(
        os.path.join(fold_dir, "val_confusion_matrix.csv")
    )
    save_confusion_matrix_png(
        cm_val, classes,
        os.path.join(fold_dir, "val_confusion_matrix.png"),
        title=f"Fold {fold} VAL Confusion Matrix (acc={val_acc_b:.3f})"
    )

    report = classification_report(
        y_val_true_b, y_val_pred_b,
        labels=list(range(len(classes))),
        target_names=classes,
        digits=4,
        zero_division=0
    )
    with open(os.path.join(fold_dir, "val_classification_report.txt"), "w") as f:
        f.write(report)

    fold_best_acc.append(float(best_val_acc))
    fold_model_paths.append(os.path.join(fold_dir, "best_model.pth"))
    all_fold_summaries.append({
        "fold": fold,
        "best_val_acc": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "n_train": int(len(train_sub)),
        "n_val": int(len(val_sub)),
    })

    print(f"\nFold {fold} summary:")
    print(f"  best_val_acc = {best_val_acc:.4f} at epoch {best_epoch}")
    print(f"  saved: {fold_dir}/best_model.pth")
    print(f"  saved: {fold_dir}/loss_curve.png, {fold_dir}/acc_curve.png, {fold_dir}/val_confusion_matrix.png")

# Save CV summary
cv_mean = float(np.mean(fold_best_acc))
cv_std  = float(np.std(fold_best_acc))

summary = {
    "k_folds": K_FOLDS,
    "fold_best_val_acc": fold_best_acc,
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "classes": classes,
    "n_total_samples": int(len(df)),
    "n_cv_pool": int(len(df_cv)),
    "n_final_test": int(len(df_test)),
    "hyperparams": {
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "epochs_max": NUM_EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTH,
        "mixup_alpha": MIXUP_ALPHA,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS
    },
    "fold_details": all_fold_summaries
}
with open(os.path.join(OUTDIR, "cv_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n==============================")
print("CROSS-VALIDATION SUMMARY")
print("==============================")
print(f"Fold best val acc: {np.round(fold_best_acc, 4).tolist()}")
print(f"CV mean(best val acc) = {cv_mean:.4f} ± {cv_std:.4f}")
print(f"Saved CV summary to: {os.path.join(OUTDIR, 'cv_summary.json')}")

# ============================
# 8) FINAL EVALUATION ON UNSEEN TEST SET
# ============================
best_fold_idx = int(np.argmax(fold_best_acc))  # 0-based index
best_model_path = fold_model_paths[best_fold_idx]

print("\n==============================")
print("FINAL TEST EVALUATION")
print("==============================")
print(f"Best fold = {best_fold_idx + 1} with best_val_acc = {fold_best_acc[best_fold_idx]:.4f}")
print(f"Loading model from: {best_model_path}")

final_model = get_model(len(classes))
final_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
final_model.eval()

test_ds = WST_S1_Dataset(df_test, class_to_idx, img_size=IMG_SIZE)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

test_criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(final_model, test_loader, test_criterion)

print(f"Test loss = {test_loss:.4f}")
print(f"Test acc  = {test_acc:.4f}")

test_report = classification_report(
    y_true, y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=4,
    zero_division=0
)
with open(os.path.join(OUTDIR, "final_test_classification_report.txt"), "w") as f:
    f.write(test_report)

cm_test = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
pd.DataFrame(cm_test, index=classes, columns=classes).to_csv(
    os.path.join(OUTDIR, "final_test_confusion_matrix.csv")
)

save_confusion_matrix_png(
    cm_test, classes,
    os.path.join(OUTDIR, "final_test_confusion_matrix.png"),
    title=f"FINAL TEST Confusion Matrix (acc={test_acc:.3f})"
)

final_metrics = {
    "best_fold": best_fold_idx + 1,
    "best_fold_val_acc": float(fold_best_acc[best_fold_idx]),
    "cv_mean_best_val_acc": cv_mean,
    "cv_std_best_val_acc": cv_std,
    "final_test_loss": float(test_loss),
    "final_test_acc": float(test_acc),
}
with open(os.path.join(OUTDIR, "final_metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=2)

print("\nSaved thesis-ready outputs to:", OUTDIR)
print("Files you will use in thesis (PNG):")
print("  - fold_*/loss_curve.png")
print("  - fold_*/acc_curve.png")
print("  - fold_*/val_confusion_matrix.png")
print("  - final_test_confusion_matrix.png")
print("Plus CSV/TXT/JSON reports for appendix.")


--- Loading metadata ---
--- Indexing NPZ files ---

--- Dataset summary (after merge + filtering) ---
Total samples: 1154
Classes kept (5): ['CT_ET_IT', 'Lamp5', 'Pvalb', 'Sst', 'Vip']

Class counts:
RNA_family
CT_ET_IT    358
Pvalb       286
Sst         270
Vip         152
Lamp5        88

Dropped classes (< MIN_SAMPLES_PER_CLASS): ['Sncg', 'NP']

--- Split summary ---
CV pool:   923
Test set:  231

FOLD 1/3
Epoch 01 | train_loss=1.1006 train_acc=0.455 | val_loss=1.0810 val_acc=0.718 | lr=3.00e-04
Epoch 02 | train_loss=0.6780 train_acc=0.511 | val_loss=0.6953 val_acc=0.831 | lr=3.00e-04
Epoch 03 | train_loss=0.6298 train_acc=0.662 | val_loss=0.6416 val_acc=0.899 | lr=3.00e-04
Epoch 04 | train_loss=0.5127 train_acc=0.587 | val_loss=0.8662 val_acc=0.734 | lr=3.00e-04
Epoch 05 | train_loss=0.5592 train_acc=0.706 | val_loss=0.7051 val_acc=0.870 | lr=3.00e-04
Epoch 06 | train_loss=0.5001 train_acc=0.678 | val_loss=0.5884 val_acc=0.860 | lr=1.50e-04
Epoch 07 | train_loss=0.6444 train_acc=

In [ ]:
# ==========================================
# 1) Mount Google Drive
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2) Define folders
# ==========================================
import os
import shutil

folders = [
    "/content/cross_val_results_Resnet18",
    "/content/cross_val_results_VGG16",
    "/content/cross_val_results_inception"
]

# Destination in Drive
DRIVE_DEST = "/content/drive/MyDrive/WST_Model_Results"
os.makedirs(DRIVE_DEST, exist_ok=True)

# ==========================================
# 3) Zip + Copy each folder
# ==========================================
for folder in folders:
    if not os.path.exists(folder):
        print(f"❌ Folder not found: {folder}")
        continue

    folder_name = os.path.basename(folder)
    zip_path = f"/content/{folder_name}.zip"

    print(f"\n🔄 Zipping {folder_name} ...")
    shutil.make_archive(folder, 'zip', folder)

    print(f"📤 Uploading {folder_name}.zip to Drive ...")
    shutil.copy(f"{folder}.zip", DRIVE_DEST)

    print(f"✅ Done: {folder_name}.zip")

print("\n🎉 All folders uploaded successfully!")
print("📁 Location in Drive:", DRIVE_DEST)

Mounted at /content/drive

🔄 Zipping cross_val_results_Resnet18 ...
📤 Uploading cross_val_results_Resnet18.zip to Drive ...
✅ Done: cross_val_results_Resnet18.zip

🔄 Zipping cross_val_results_VGG16 ...
📤 Uploading cross_val_results_VGG16.zip to Drive ...
✅ Done: cross_val_results_VGG16.zip

🔄 Zipping cross_val_results_inception ...
📤 Uploading cross_val_results_inception.zip to Drive ...
✅ Done: cross_val_results_inception.zip

🎉 All folders uploaded successfully!
📁 Location in Drive: /content/drive/MyDrive/WST_Model_Results
